# 04 V2 — Gemini Schema Generation

Generates schema.org JSON-LD for each page using **Gemini 2.5 Flash**.

- Input: `data/processed/pages_for_generation.jsonl` (HTML + screenshots)
- Output: `data/generated/*.json` (one per page, with validation metadata)
- Cost: ~$10 for 7.5K pages (standard pricing)
- Time: ~2.5 hours at 50 pages/min

**Requires**: `GEMINI_API_KEY` in `.env`, `pip install google-genai`

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv('../.env', override=True)
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY in .env'
print('API key loaded')

In [ ]:
# Dry run — check page count and cost estimate
!python3 -u ../scripts/gemini_generate.py --dry-run

In [ ]:
# Test batch (20 pages) to verify quality before full run
!python3 -u ../scripts/gemini_generate.py --limit 20 --workers 4

In [ ]:
# Quick spot-check of test results
import json, glob
from collections import Counter

files = glob.glob('../data/generated/*.json')
valid = sum(1 for f in files if json.load(open(f))['valid'])
print(f'{valid}/{len(files)} valid ({valid/len(files)*100:.0f}%)')

if valid / max(len(files), 1) < 0.85:
    print('WARNING: Valid rate below 85% — check errors before full run!')
else:
    print('Looks good — proceed to full run')

In [ ]:
# Full run — generates all remaining pages (skips already-done)
# Takes ~2.5 hours for 7.5K pages at 16 workers
!python3 -u ../scripts/gemini_generate.py --workers 16

In [ ]:
# Final results summary
import json, glob
from collections import Counter

files = glob.glob('../data/generated/*.json')
results = [json.load(open(f)) for f in files]
valid = [r for r in results if r['valid']]

print(f'Total: {len(results):,}  Valid: {len(valid):,} ({len(valid)/len(results)*100:.1f}%)')
print(f'\nType breakdown:')
types = Counter(r['schema_types'][0] for r in valid if r['schema_types'])
for t, n in types.most_common(15):
    print(f'  {t:30s} {n:5,}')

print(f'\nNext: 04.5_v2_generation_review.ipynb for visual QA')